## install the packages

In [ ]:
! pip install scikit-learn==1.6.0 streamlit==1.30.0 pandas==2.0.3 numpy==1.26.4 matplotlib==3.8.2 seaborn==0.12.3 plotly==5.17.0

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import streamlit as st


In [ ]:
df = pd.read_csv("IKEA_product_catalog-2.csv")
df.head()

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
df.describe()

In [ ]:
df=df[['product_id', 'product_name', 'product_type', 'price', 'main_category', 'sub_category', 'product_rating', 'product_rating_count', 'currency', 'product_measurements', 'country' ]]
df.head()

In [ ]:
df_new=df.copy()

## clean data

In [ ]:
df_new.isna().sum()

In [ ]:
df_new.info()

In [ ]:
df_new[['price', 'product_rating', 'product_rating_count']].head(10)

In [ ]:
df_new['currency'].value_counts()

In [ ]:
df_new.isna().sum()

In [ ]:
df_new.duplicated().sum()

In [ ]:
df_new.dtypes

In [ ]:
df_new[['product_rating', 'product_rating_count']].head(20)

In [ ]:
df_new['product_rating'] = pd.to_numeric(df_new['product_rating'], errors='coerce')

df_new['product_rating_count'] = pd.to_numeric(df_new['product_rating_count'], errors='coerce')

In [ ]:
df_new.isna().sum()

In [ ]:
df_new.dtypes

In [ ]:
df_new.isna().mean() * 100

In [ ]:
df_new.shape

In [ ]:
df_clean = df_new.dropna(
    subset=['product_rating', 'product_rating_count']
).copy()

In [ ]:
df_clean.describe()

In [ ]:
df_new = df_new.dropna(subset=['product_rating', 'product_rating_count'])

In [ ]:
df_new.isna().sum()

In [ ]:
df_new.shape

In [ ]:
sns.pairplot(df_new.sample(5000, random_state=42),
             vars=['price', 'product_rating', 'product_rating_count'])
plt.show()

In [ ]:
sns.pairplot(df_new, hue='main_category', palette='Set1')
plt.show()

In [ ]:
plt.figure(figsize=(15, 6))
sns.countplot(x='product_rating', data=df_new, color='steelblue')
plt.xticks(rotation=90)
plt.show()


In [ ]:
px.histogram(df_new, x='product_rating', marginal='box',
             nbins=30, title='Product Rating Distribution')

In [ ]:
px.histogram(df_new, x='product_rating_count',
             marginal='box', nbins=30,
             title='Product Review Count Distribution')

## input feature selection

In [ ]:
X = df_new[['price', 'product_rating', 'product_rating_count']]

In [ ]:
X

In [ ]:
X['product_rating'].value_counts()

## split data into train and test

In [ ]:
x_train, x_test = train_test_split(X, test_size=0.2, random_state=42)

## Apply Scaling 

In [ ]:
sc = StandardScaler()
x_train_scaled = sc.fit_transform(x_train)
x_test_scaled = sc.transform(x_test)

## Apply the modeling 

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

km = KMeans(n_clusters=3, random_state=42, n_init=10)

km.fit(x_train_scaled)

train_clusters = km.predict(x_train_scaled)
test_clusters = km.predict(x_test_scaled)

print("Training Silhouette Score:",
      silhouette_score(x_train_scaled, train_clusters, sample_size=5000, random_state=42))

print("Testing Silhouette Score:",
      silhouette_score(x_test_scaled, test_clusters, sample_size=5000, random_state=42))

In [ ]:
for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(x_train_scaled)

    labels = km.predict(x_train_scaled)

    score = silhouette_score(x_train_scaled, labels,
                             sample_size=5000, random_state=42)

    print("Number of clusters:", k, "Silhouette Score:", score)

In [ ]:
from sklearn.metrics import davies_bouldin_score, calinski_harabasz_score

print("Davies-Bouldin Index:",
      davies_bouldin_score(x_test_scaled, test_clusters))

print("Calinski-Harabasz Score:",
      calinski_harabasz_score(x_test_scaled, test_clusters))

In [ ]:
from sklearn.mixture import GaussianMixture

gmm = GaussianMixture(n_components=3, random_state=42)

gmm.fit(x_train_scaled)

gmm_train_clusters = gmm.predict(x_train_scaled)
gmm_test_clusters = gmm.predict(x_test_scaled)

print("Training Silhouette Score:",
      silhouette_score(x_train_scaled, gmm_train_clusters,
                       sample_size=5000, random_state=42))

print("Testing Silhouette Score:",
      silhouette_score(x_test_scaled, gmm_test_clusters,
                       sample_size=5000, random_state=42))

## cross validation

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline

def clustering_score(model, X):
    labels = model.predict(X)
    return silhouette_score(X, labels, sample_size=5000,
                            random_state=42)

km_pipeline = make_pipeline(StandardScaler(), km)
gmm_pipeline = make_pipeline(StandardScaler(), gmm)

km_cv = cross_val_score(km_pipeline, x_train, cv=5,
                       scoring=clustering_score)

gmm_cv = cross_val_score(gmm_pipeline, x_train, cv=5,
                        scoring=clustering_score)

print("K-Means CV Score:", km_cv.mean())
print("GMM CV Score:", gmm_cv.mean())

## hyper parameter tuning

In [ ]:
from sklearn.model_selection import GridSearchCV

def clustering_score(model, X):
    X_scaled = model.steps[0][1].transform(X)
    labels = model.predict(X)
    return silhouette_score(X_scaled, labels,
                            sample_size=5000, random_state=42)

In [ ]:
km_grid = GridSearchCV(
    km_pipeline,
    param_grid={'kmeans__n_clusters': [2, 3, 4, 5, 6]},
    cv=5,
    scoring=clustering_score
)

km_grid.fit(x_train)

print("Best Parameters:", km_grid.best_params_)
print("Best Score:", km_grid.best_score_)

In [ ]:
gmm_grid = GridSearchCV(
    gmm_pipeline,
    param_grid={'gaussianmixture__n_components': [2, 3, 4, 5, 6]},
    cv=5,
    scoring=clustering_score
)

gmm_grid.fit(x_train)

print("Best Parameters:", gmm_grid.best_params_)
print("Best Score:", gmm_grid.best_score_)

## create the pipline for deployment

In [ ]:
df


In [ ]:
df.isna().sum()

## split input and output

In [ ]:
X = df_new[['price', 'product_rating', 'product_rating_count']]

In [ ]:
x_train, x_test = train_test_split(X, test_size=0.2, random_state=42)

In [ ]:
numerical_columns = x_train.select_dtypes(include=['int64', 'float64']).columns
numerical_columns

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

numerical_pipeline

In [ ]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer([
    ('num', numerical_pipeline, numerical_columns)
])

preprocessor

In [ ]:
from sklearn.base import clone

final_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', clone(km_grid.best_estimator_.named_steps['kmeans']))
])

final_pipeline.fit(x_train)

final_pipeline

In [ ]:
x_train

In [ ]:
final_pipeline.fit(x_train)

In [ ]:
y_pred = final_pipeline.predict(x_test)
y_pred

In [ ]:
pd.Series(y_pred).value_counts()

In [ ]:
import joblib

joblib.dump(final_pipeline, 'ikea_pipeline.pkl')

In [ ]:
clustering = joblib.load('ikea_pipeline.pkl')

In [ ]:
clustering.predict(pd.DataFrame({
    'price': [150],
    'product_rating': [4.5],
    'product_rating_count': [100]
}))

In [ ]:
%%writefile app.py
# Streamlit application

import streamlit as st
import pandas as pd
import joblib
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

df = pd.read_csv("IKEA_product_catalog-2.csv")

st.title("IKEA Product Clustering Application")
st.plotly_chart(px.histogram(df, x='product_rating', marginal='box', nbins=30, title='Product Rating Distribution'))



clustering = joblib.load('ikea_pipeline.pkl')


def predict_cluster(price, product_rating, product_rating_count):
    input_data = pd.DataFrame({
        'price': [price],
        'product_rating': [product_rating],
        'product_rating_count': [product_rating_count]
    })

    prediction = clustering.predict(input_data)

    return prediction[0]


def main():
    st.title('IKEA Product Clustering')

    html_temp = """
    <div style="background-color:#0058A3;padding:10px">
        <h2 style="color:#FFDA1A;text-align:center;">
            IKEA Product Segmentation App
        </h2>
    </div>
    """

    st.markdown(html_temp, unsafe_allow_html=True)

    price = st.number_input("Product Price", min_value=0.0, value=150.0)

    product_rating = st.number_input("Product Rating", min_value=1.0, max_value=5.0, value=4.5, step=0.1)

    product_rating_count = st.number_input("Number of Reviews", min_value=1, value=100)

    if st.button("Predict Cluster"):
        result = predict_cluster(price, product_rating, product_rating_count)
        st.success(f"Predicted Cluster: {result}")


if __name__ == '__main__':
    main()

In [ ]:
%pip install streamlit

In [ ]:
import sklearn
print(sklearn.__version__)

In [ ]:
df[['price', 'product_rating', 'product_rating_count']].to_csv(
    'IKEA_product_catalog-2.csv', index=False
)

In [88]:
pd.Series(final_pipeline.predict(x_test)).value_counts()

1    52816
0     1002
Name: count, dtype: int64

In [89]:
cluster_data = x_test.copy()
cluster_data['Cluster'] = final_pipeline.predict(x_test)

cluster_data.groupby('Cluster').mean()

,price,product_rating,product_rating_count
Cluster,,,
0,1774.060719,4.553792,3688.348303
1,12577.455113,4.456483,113.134448
